# Практическая работа: анализ данных пассажиров «Титаника»



## Основные понятия

- **Series** — одномерная структура Pandas с индексами.
- **DataFrame** — двумерная таблица, состоящая из столбцов Series.
- **Индексация** — обращение к строкам, столбцам и отдельным элементам.
- **Фильтрация** — выбор строк по логическим условиям.
- **NaN** — обозначение отсутствующего значения.

Импортируем Pandas и NumPy и выполним остальные пункты задания.

In [ ]:

import pandas as pd
import numpy as np

# Загрузка исходного набора данных
file_path = "titanic.csv"
df = pd.read_csv(file_path)

print("ЧАСТЬ II. ПЕРВИЧНЫЙ АНАЛИЗ")
n = int(input("Сколько первых строк показать? "))
display(df.head(n))
display(df.tail(5))
print("Размер таблицы:", df.shape)
print("Столбцы:", df.columns.tolist())
print("\nТипы данных:")
print(df.dtypes)

# Разделение признаков по смыслу
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked", "Survived"]
print("\nЧисловые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

print("\nОписательная статистика числовых признаков:")
display(df[numeric_features].describe().T)
print("\nКоличество пассажиров каждого класса:")
print(df["Pclass"].value_counts().sort_index())
print("\nКоличество уникальных значений Embarked:", df["Embarked"].nunique())
print("Уникальные значения Embarked:", df["Embarked"].unique())

print("\nЧАСТЬ III. ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ")
print("Количество пропусков:")
print(df.isna().sum())
print("\nПроцент пропусков:")
print((df.isna().mean() * 100).round(2))
print("Столбец с максимальным количеством пропусков:", df.isna().sum().idxmax())

age_before = df["Age"].agg(["mean", "median", "min", "max", "std", "count"])
print("\nВозраст ДО заполнения:")
print(age_before)

age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)
df = df.dropna(subset=["Embarked"])

cabin_missing_percent = df["Cabin"].isna().mean() * 100
print(f"\nДоля пропусков Cabin: {cabin_missing_percent:.2f}%")
if cabin_missing_percent > 50:
    df = df.drop(columns=["Cabin"])
    print("Столбец Cabin удалён: пропусков больше 50%.")
else:
    print("Столбец Cabin оставлен.")

age_after = df["Age"].agg(["mean", "median", "min", "max", "std", "count"])
age_comparison = pd.DataFrame({"До обработки": age_before, "После обработки": age_after})
print("\nСравнение возраста:")
display(age_comparison)
print("Оставшиеся пропуски:")
print(df.isna().sum())
print("Размер до обработки:", (pd.read_csv(file_path).shape))
print("Размер после обработки:", df.shape)

print("\nЧАСТЬ IV. АНАЛИЗ С ПОМОЩЬЮ NUMPY")
survived = df["Survived"].to_numpy()
print(f"Общий процент выживших: {np.mean(survived == 1) * 100:.2f}%")

for sex in df["Sex"].unique():
    part = df[df["Sex"] == sex]
    percent = np.mean(part["Survived"].to_numpy() == 1) * 100
    print(f"Выживаемость ({sex}): {percent:.2f}%")

print("\nСредний возраст по полу:")
for sex in df["Sex"].unique():
    ages = df.loc[df["Sex"] == sex, "Age"].to_numpy()
    print(f"{sex}: {np.mean(ages):.2f}")

print("\nСредний возраст выживших и погибших по полу:")
for sex in df["Sex"].unique():
    for status in [1, 0]:
        ages = df.loc[(df["Sex"] == sex) & (df["Survived"] == status), "Age"].to_numpy()
        if len(ages):
            label = "выжившие" if status == 1 else "погибшие"
            print(f"{sex}, {label}: {np.mean(ages):.2f}")

fares = df["Fare"].dropna().to_numpy()
print(f"\nСтоимость билета: минимум {np.min(fares):.2f}, максимум {np.max(fares):.2f}, среднее {np.mean(fares):.2f}")
print(f"Стандартное отклонение возраста (NumPy): {np.std(df['Age'].to_numpy(), ddof=1):.2f}")

print("\nЧАСТЬ V. ФИЛЬТРАЦИЯ")
task1 = df[(df["Sex"] == "male") & (df["Age"] > 30) & (df["Pclass"] == 1)]
print("Мужчины старше 30 лет первого класса:", len(task1))
display(task1.head(10))

task2 = df[((df["Age"] < 18) | (df["Sex"] == "female")) & (df["Survived"] == 1)]
print("Пассажиры младше 18 лет или женщины, которые выжили:", len(task2))

class3_mean_fare = df.loc[df["Pclass"] == 3, "Fare"].mean()
task3 = df[(df["Pclass"] == 3) & (df["Fare"] > class3_mean_fare)]
print(f"Пассажиры 3 класса с билетом дороже среднего ({class3_mean_fare:.2f}):", len(task3))
display(task3.head(10))

task4 = df[(df["Age"].between(20, 40)) & (df["Embarked"] == "S")]
print("Пассажиры 20–40 лет из порта S:", len(task4))
display(task4.head(10))

print("\nЧАСТЬ VI. ГРУППИРОВКА И АГРЕГИРОВАНИЕ")
statistics = df.groupby(["Pclass", "Sex"]).agg(
    passenger_count=("PassengerId", "count"),
    mean_age=("Age", "mean"),
    survival_percentage=("Survived", lambda x: x.mean() * 100),
    mean_fare=("Fare", "mean")
).reset_index()
display(statistics)

embarked_statistics = df.groupby("Embarked").agg(
    passenger_count=("PassengerId", "count"),
    survival_percentage=("Survived", lambda x: x.mean() * 100)
).reset_index()
display(embarked_statistics)

survival_by_class = df.groupby("Pclass")["Survived"].mean() * 100
print("Доля выживших по классам (%):")
print(survival_by_class)
print("Наибольшая доля выживших — класс:", survival_by_class.idxmax())

print("\nЧАСТЬ VII. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
df.to_csv("titanic_processed.csv", index=False)
statistics.to_csv("titanic_statistics.csv", index=False)
embarked_statistics.to_csv("embarked_statistics.csv", index=False)
reloaded = pd.read_csv("titanic_processed.csv")
print("Обработанный CSV повторно загружен.")
print("Размер повторно загруженного файла:", reloaded.shape)
print("Совпадает ли размер с исходным обработанным DataFrame:", reloaded.shape == df.shape)


## Контрольные вопросы и выводы

**Почему Pclass — категориальный признак?** Числа 1, 2 и 3 обозначают классы обслуживания, а не количественную величину. Разность между классами не имеет смысла как арифметическая разница.

**Почему медиана предпочтительнее среднего для заполнения Age?** Медиана меньше зависит от очень маленьких и очень больших значений возраста.

**Какие характеристики возраста изменились?** После заполнения пропусков медианой увеличивается число известных значений. Среднее и стандартное отклонение могут измениться; медиана, минимум и максимум обычно остаются прежними, если добавленное значение находится внутри исходного диапазона.

**Сохраняется ли исходное распределение возрастов?** Нет. Заполнение всех пропусков одним значением увеличивает частоту медианного возраста и меняет форму распределения.

**Различие И и ИЛИ при фильтрации:** И (`&`) требует выполнения всех условий одновременно, ИЛИ (`|`) — хотя бы одного условия.

**Группировка и фильтрация:** фильтрация выбирает строки по условию, группировка объединяет строки с одинаковыми значениями признака и рассчитывает показатели для каждой группы.

### Выводы по данным

В исходной таблице больше всего пропусков в `Cabin` и `Age`. Пропуски `Age` заполнены медианой исходных известных возрастов. Столбец `Cabin` удаляется, если доля пропусков превышает 50%. Строки без `Embarked` удаляются.

Важное ограничение предоставленного набора: в нём все женщины имеют `Survived = 1`, а все мужчины — `Survived = 0`. Поэтому рассчитанные проценты выживаемости отражают именно особенности этого файла и не должны трактоваться как универсальная статистика по пассажирам «Титаника».

Наиболее полезными инструментами оказались `head()`, `info()`, `isna()`, `fillna()`, логическая фильтрация, `groupby()`, `agg()` и функции NumPy `mean()`, `std()`, `min()` и `max()`.